# Step 17.5 — Uncompiled Training Step

## 把Forward、Loss、逆向自动微分、Adam更新与显式求值连接成一个单步

17.1–17.2准备了语言模型的scalar loss，17.3解释了 `nn.value_and_grad` 返回的参数梯度树，17.4解释了Adam与优化器状态。本节按既有roadmap连接这五个环节，准备一个普通Python单步函数。

材料审核方已独立核验17.4实际母版、文档、固定MLX接口和完整教学回复，核心静态PASS，实际修正 **0/2**。这是材料验收，不是学员掌握或运行反馈。17.5本次初始生成实际修正为 **0/2**。

本母版全部代码单元均未执行、无输出。只编写发布材料，没有执行Notebook、测试、训练、实验、采样或用户work副本，没有读取/下载语料，也不代答练习。**本轮止于17.5，即大Step17末节，不进入大Step18。**


## 0. 新 Work Copy：本节完整恢复必要定义

沿用项目 Python 3.12 / `.venv` kernel 与固定 `mlx==0.32.2`，只需项目已经安装的MLX：

- `mlx.core as mx`：数组、broadcast、reshape、mean与本节的显式求值接口；
- `mlx.nn as nn`：Module、交叉熵与参数求导变换；
- `mlx.optimizers as optim`：Adam优化器。`as` 给模块指定一个短名称。

新副本自带17.3的完整小Module、toy chunks与X/Y、loss函数，随后创建同一组model、梯度函数和Adam，再定义单步。以后使用时，从本节imports开始按顺序使用即可；无需旧kernel、旧Notebook的 `%run`、语料缓存、sampler或完整MiniGPT代码。

17.4用手填梯度研究优化器，本节则准备在单步内部调用自动微分来取得梯度；不会把那组手填 `example_grads` 冒充成当前参数处的计算结果。单步代码只被写入母版，本轮不调用它。


In [ ]:
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

# Imports for a future work copy. All cells in this master are unexecuted.


## 1. 一个Training Step做什么？先区分参数的前后版本

**Training step（训练单步）** 使用一批给定X/Y，从当前参数得到loss和梯度，再进行一次参数更新。这里一批有B×T个token-level targets，但只有一次优化器更新，不是每个target各更新一次。

本节的 **uncompiled（未编译）** 指这个单步是普通Python函数，没有使用 `mx.compile` 包装。它仍使用MLX数组运算和自动微分；不能据此认为每一行Python都立即完成了全部数值计算。编译机制与性能比较不在本节展开。

把参数统称为 \(\theta\)，把Adam保存的step、m、v等统称为优化器状态。用 \(k\) 表示第几次更新，而不是token位置t。第k次调用中：

| 环节 | 数学/接口含义 |
|---|---|
| **Forward，前向计算** | 当前参数 \(\theta_{k-1}\) 与X产生raw logits |
| **Loss，损失** | logits与已shift的Y产生scalar \(L(\theta_{k-1};X,Y)\) |
| **Reverse-mode autodiff，逆向自动微分** | 得到 \(g_k=\nabla_\theta L(\theta_{k-1};X,Y)\) |
| **Optimizer update，优化器更新** | 用g与历史状态形成 \(\theta_k\) 及新的m/v/step |
| **Explicit evaluation boundary，显式求值边界** | 要求本次loss、新参数和新优化器状态完成求值 |

这是计算与状态的依赖关系，不是逐行保证设备任务已经完成的时间表。

特别留意：本次返回的loss来自更新前的 \(\theta_{k-1}\)；调用结束后的model则持有更新后的 \(\theta_k\)。这两个时点会在函数名、变量名和手算说明中明确区分。


## 2. Forward与Scalar Loss：恢复一个能手算的完整例子

继续使用17.3的 `SharedLogitModel`。必要Python与MLX含义简要恢复：

- `class ... (nn.Module)` 定义继承Module的类；`super().__init__()` 初始化父类；`self` 是当前实例。
- `self.head` 是公开的 **dict（字典）**，`["bias"]` 根据键名读取数组叶子；MLX按参数树管理这片浮点数组。
- `__call__` 定义实例写成 `model(X)` 时调用的规则，这就是本例的forward。
- `B, T = X.shape` 按顺序取出两个维度；`mx.broadcast_to` 把共享向量 `(V,)` 用在每个位置，形成 `(B,T,V)`。

本例只有3个共享参数。所有位置使用同一组raw scores；它只读取X的shape，不读取token内容或上下文。它是单步接口的toy替身，不是MiniGPT，也不是为MiniGPT推荐全零初始化。

数据仍用17.3的两行明确给定的chunks。`mx.int32` 表示整数token IDs；`[:, :-1]` 取输入，`[:, 1:]` 取右移一个位置的目标。X/Y是模型外的数据，不把它们注册为参数。


In [ ]:
class SharedLogitModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = {
            "bias": mx.array([0.0, 0.0, 0.0], dtype=mx.float32),
        }

    def __call__(self, X):
        B, T = X.shape
        V = self.head["bias"].shape[0]
        return mx.broadcast_to(self.head["bias"], (B, T, V))


# Definition only. This toy uses X.shape, not token content or context.


In [ ]:
# EXPLICIT TOY DATA — prepared only, NOT executed.

toy_chunks = mx.array([[0, 2, 1], [1, 0, 2]], dtype=mx.int32)
X = toy_chunks[:, :-1]  # [[0, 2], [1, 0]], shape (2, 2)
Y = toy_chunks[:, 1:]   # [[2, 1], [0, 2]], shape (2, 2), already shifted

# B=2, T=2, V=3, N=B*T=4. No corpus, sampler, or RNG call.


### 在loss_fn内部完成forward与loss

Python的 `def` 定义函数，函数体到调用时才使用传入的实参，`return` 返回结果。下面的 `loss_fn(model, X, Y)` 恢复17.2–17.3的接口：

| 运算 | 本例shape |
|---|---|
| 参数叶子 `head.bias` | `(3,)` |
| `logits = model(X)` | `(2,2,3)` |
| 配对reshape后的logits / Y | `(4,3)` / `(4,)` |
| `cross_entropy(..., reduction="none")` | `(4,)` |
| `mx.mean(token_losses)` | `()` |

N=B×T表示监督位置总数；reshape两侧保持相同的位置顺序，展平Y为 `[2,1,0,2]`。最后一轴V是类别轴，所以交叉熵用 `axis=-1`，直接接收raw logits与整数targets。Y已经shift，不再次右移。

`mx.mean` 对四项位置loss等权平均。该平均会自然进入梯度，优化器处不再除以N。返回值保持 **scalar MLX array（标量MLX数组）**，shape `()`；不要在loss函数中用 `.item()` 或 `float(...)` 把它变成Python数。

forward必须在loss函数内部依赖当前模型参数。以后调用变换后的loss函数时，这条forward与loss关系已经包含在里面，不需要先额外调用一次模型或loss再重复求导。


In [ ]:
def loss_fn(model, X, Y):
    logits = model(X)
    B, T, V = logits.shape
    N = B * T

    flat_logits = mx.reshape(logits, (N, V))
    flat_targets = mx.reshape(Y, (N,))
    token_losses = nn.losses.cross_entropy(
        flat_logits,
        flat_targets,
        axis=-1,
        reduction="none",
    )
    return mx.mean(token_losses)


# Definition only. Return a scalar MLX array; no item()/float() conversion here.


## 3. Reverse-mode Autodiff：从一个Loss把敏感度传回参数

17.3已经解释参数梯度的含义。本节再看它如何沿现有计算关系传回来。\(\partial L/\partial a\) 表示L对中间量a变化的局部敏感度。

**Reverse-mode autodiff（逆向自动微分）** 从标量输出出发，沿前向计算的依赖关系反向应用求导规则。回顾 **chain rule（链式法则）**：沿一条依赖路径连接局部导数时相乘；同一个量通过多条路径影响loss时，把各路径贡献相加。

它不是把token顺序反过来，不是求前向运算的逆函数，也不是逐个扰动参数的有限差分。我们继续让MLX完成求导，不手写一套backward实现。

### 沿本例的mean、交叉熵和共享参数理解反向贡献

定义r为展平后的监督位置，v为类别编号，\(y_r\) 为该位置正确类别。\(z_{r,v}\) 是logit，\(p_{r,v}\) 是它在类别轴softmax后的概率；\(q_{r,v}\) 在v等于正确类别时为1，否则为0。q只是手算中的one-hot指示量，代码继续传整数Y。

位置损失为 \(\ell_r\)，总loss为：

$$
L=\frac1N\sum_{r=0}^{N-1}\ell_r,\qquad N=4.
$$

从输出本身开始：

$$
\frac{\partial L}{\partial L}=1.
$$

mean把敏感度传给每个位置：

$$
\frac{\partial L}{\partial\ell_r}=\frac1N.
$$

17.3的交叉熵局部导数为 \(\partial\ell_r/\partial z_{r,v}=p_{r,v}-q_{r,v}\)。乘上mean传来的1/N：

$$
\frac{\partial L}{\partial z_{r,v}}
=\frac1N\left(p_{r,v}-q_{r,v}\right).
$$

这一级敏感度按展平logits组织时有 `(N,V)` 个分量，本例为 `(4,3)`。reshape只改变组织方式，两侧位置关系继续保持。

本例每个位置共用同一个参数分量：\(z_{r,v}=\theta_v\)。同一参数通过四个位置影响L，所以最后要把这些路径的贡献相加：

$$
\frac{\partial L}{\partial\theta_v}
=\frac1N\sum_{r=0}^{N-1}\left(p_{r,v}-q_{r,v}\right).
$$

这就回到 `(3,)` 的参数梯度，而不是给共享参数留下四份独立更新。

零初始参数下，每个类别概率为1/3；targets `[2,1,0,2]` 中三个类别出现比例为1/4、1/4、1/2，于是：

$$
g_1=\left[\frac13-\frac14,\frac13-\frac14,\frac13-\frac12\right]
=\left[\frac1{12},\frac1{12},-\frac16\right].
$$

这是手算解释，不是本轮执行自动微分所得的输出。代码中的grads会由单步内的变换调用取得，不把这个手算数组硬填进更新。


### 用nn.value_and_grad表示这条求导关系

**Function object（函数对象）** 是函数本身：`loss_fn` 可以传给另一个函数；`loss_fn(model, X, Y)` 则是调用它。**Function transformation（函数变换）** 接收一个函数并返回新的可调用对象。

固定MLX v0.32.2的 `nn.value_and_grad(model, loss_fn)` 指定对该model的可训练参数求导，并返回一个函数，我们命名为 `loss_and_grad_fn`。以后在单步内部调用：

```python
loss_before_update, grads = loss_and_grad_fn(model, X, Y)
```

左侧是 **tuple unpacking（元组解包）**：第一项接收scalar loss，第二项接收对应可训练参数树的梯度。本例 `grads["head"]["bias"]` 的shape为 `(3,)`。

构造变换时的model选择求导参数；调用时的model、X、Y则是转交给我们自己定义的 `loss_fn(model, X, Y)` 的实参，包装器不会自动插入model。两处必须使用同一个实例。包装器每次调用取得当前可训练参数，不是固定在初始参数值。

本节直接使用返回的梯度树，不添加 `.backward()` 或 `.zero_grad()`。求导变换内部会把本次求导使用的参数接入Module；它本身不执行Adam学习更新。固定接口依据是 [v0.32.2 nn/utils.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/utils.py)。


## 4. Optimizer Update：把新参数装回同一个Module

17.4中的 `apply_gradients(gradients, parameters)` 返回新参数树。本节用与Module连接的接口：

```python
optimizer.update(model, grads)
```

**Method（方法）** 是对象上的函数。这个方法接收model和梯度树，将Adam算出的新参数安装进该模型，并推进优化器自己的m、v、step。它有 **side effects（对对象状态的修改）**，返回值为 `None`，不返回新模型。不要写 `model = optimizer.update(...)`，否则会把model变量赋成None。

调用后仍是原来的model对象，但其中参数叶子已被替换为新的数组；下一次forward会读取这些新参数。`optimizer.state` 则保存更新后的优化器状态。

### 初始化整组对象：放在单步函数外

这里一次创建同一组model、求导函数与optimizer。`model.trainable_parameters()` 给出可训练参数树；`optimizer.init(...)` 在这个全新对象上建立零初始m/v，step仍为0，没有学习更新。不要在每次单步内部重新构造Adam，否则会丢掉历史。

Adam延续17.4的明确选择：

| 设置 | 本节值 | 固定MLX v0.32.2默认行为 |
|---|---|---|
| `learning_rate` | `1e-3`，即0.001 | 必填，无默认数值 |
| `betas` | `[0.9, 0.999]` | `[0.9, 0.999]` |
| `eps` | `1e-8` | `1e-8` |
| `bias_correction` | **`True`** | **`False`** |

`name=value` 是关键字实参，`betas` 列表两项对应一阶矩与二阶矩的系数，`True` 是开启偏差修正的布尔值。这里只延续已解释的教学配置，不把它说成已调优的MiniGPT设置。

固定版本的更新与状态契约见 [v0.32.2 optimizers.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/optimizers/optimizers.py)。


In [ ]:
# ONE MATCHED SETUP GROUP — prepared only, NOT executed.

model = SharedLogitModel()
loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

optimizer = optim.Adam(
    learning_rate=1e-3,
    betas=[0.9, 0.999],
    eps=1e-8,
    bias_correction=True,  # Explicit lesson choice; the MLX default is False.
)
optimizer.init(model.trainable_parameters())

# Keep this model, its gradient function, and this optimizer together.
# Setup has not performed a training update; the initial optimizer step is 0.


## 5. Explicit Evaluation Boundary：这个单步要完成哪些结果？

本节只需要理解MLX的一个执行特点：数组运算可以先记录“结果依赖哪些输入及运算”，数值工作不一定在对应Python语句走过时全部完成。这叫 **lazy evaluation（惰性求值）**。这里仅铺垫单步所需的接口含义，不展开大Step18中的图增长、调度、编译或性能专题。

**Evaluation（求值）** 指按这些依赖关系取得数组数值，不是“验证集评估”这个训练任务阶段。显式边界用：

```python
mx.eval(loss_before_update, model.parameters(), optimizer.state)
```

固定接口 `mx.eval(*args)` 中，`*args` 表示可以接收多个实参。每个实参可以是单个MLX数组，也可以是由dict、list、tuple组成的数组树；树中非数组叶子会被忽略。`model.parameters()` 和 `optimizer.state` 已是这样的树，所以无需手工逐个列出数组叶子。

这个调用要求指定数组及其依赖完成求值，等待相应结果可用后返回。它的返回值为None，不是新的loss；不要写 `loss_before_update = mx.eval(...)`。

| 显式列出的对象 | 本单步希望完成的结果 |
|---|---|
| `loss_before_update` | 本次更新前forward得到的scalar loss |
| `model.parameters()` | optimizer.update之后模型当前持有的新参数 |
| `optimizer.state` | 新的m、v、step等优化器数组状态 |

只要求loss求值，不能据此认定新的参数与全部优化器状态也已完成。把三类结果一起列出，清楚地规定本单步返回前要完成的内容；这不是对所有无关设备工作的全局同步承诺。

这里没有把grads作为额外返回值；它已参与参数和状态更新所需的依赖计算。公式上的forward、反向求导与更新，以及数组何时完成求值，是需要同时理解但不能混为一谈的两层关系。

[固定v0.32.2 Python求值接口](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/transforms.cpp)与[核心求值实现](https://github.com/ml-explore/mlx/blob/v0.32.2/mlx/transforms.cpp)支持上述调用与等待语义。在线API页面可能随版本更新，课程以固定版本为准。


### 现在才把五个环节写入一个函数

`train_step` 显式接收model、X、Y、求导函数和optimizer。Python可以把函数对象作为实参传入，所以参数 `loss_and_grad_fn` 在函数体内仍可被调用。传入model和optimizer不会自动复制它们；对这些对象的状态修改会在调用之后保留。

这里构造求导函数与Adam的代码已经放在函数外。函数体只调用一次 `loss_and_grad_fn`，其中包含forward、scalar loss以及相应的逆向自动微分；不先另外做一次forward/loss。

### 返回loss的时点

`loss_before_update` 与grads来自同一次、同一组更新前参数。之后 `optimizer.update` 把模型当前的参数引用换成新数组，但不会把已建立的loss改成“用新参数再前向一次”的loss。即使 `mx.eval` 放在update之后，返回的仍是：

$$
L(\theta_{k-1};X,Y),
$$

而此时model持有的是 \(\theta_k\)。因此这个返回值不能被称为“更新后loss”，也不能用于声称更新已经使loss下降。想得到新参数的loss需要另一次forward/loss；本节不追加这个调用。

`def` 只定义单步函数，不会自动调用它。下一个代码单元仍是未执行的母版内容。


In [ ]:
def train_step(model, X, Y, loss_and_grad_fn, optimizer):
    # Forward + scalar loss + reverse-mode parameter gradients.
    loss_before_update, grads = loss_and_grad_fn(model, X, Y)

    # Install new parameters in this model; advance Adam's state.
    optimizer.update(model, grads)

    # Complete the requested loss, updated parameters, and optimizer state.
    mx.eval(loss_before_update, model.parameters(), optimizer.state)

    # This loss belongs to the forward BEFORE this call's parameter update.
    return loss_before_update


# Ordinary Python function, without mx.compile. Definition only, not executed.


### 首次调用的手算解释：Loss与状态属于不同的时点

对于本节全新对象组，初始 \(\theta_0=[0,0,0]\)，m/v为零，step为0。沿17.3–17.4已经解释的数学：

| 对象 | 首次调用中的手算值 | Shape |
|---|---|---|
| 返回的更新前loss | \(\ln3\) | `()` |
| 本次参数梯度g₁ | `[1/12, 1/12, -1/6]` | `(3,)` |
| 更新后的m₁ | `[1/120, 1/120, -1/60]` | `(3,)` |
| 更新后的v₁ | `[1/144000, 1/144000, 1/36000]` | `(3,)` |
| 更新后的step | `1` | `()` |

因为显式开启bias correction，第一步的修正矩为 \(\hat m_1=g_1\)、\(\hat v_1=g_1^2\)。新参数的精确实数表达式为：

$$
\theta_1=-0.001\,
\frac{[1/12,\ 1/12,\ -1/6]}
{[1/12,\ 1/12,\ 1/6]+10^{-8}}.
$$

仅为看尺度而忽略极小的epsilon时，约为 `[-0.001, -0.001, +0.001]`。实际准备代码保留epsilon，并会有float32舍入。这些都是推导预期，没有运行验证；我们没有计算或比较更新后的loss。

### 一次调用后的日志：最后才把scalar交给Python

**Logging（日志显示）** 可以在函数返回后读已有结果。`array.item()` 从只有一个元素的MLX数组取得普通Python数值；本例返回loss与step都是shape `()`。它本身会要求该数组求值，因此不能把只读取loss当作整步参数/state的求值边界。

我们已经把所需三类结果的 `mx.eval` 放在函数内；下面 `.item()` 只用于显示返回值。它没有进入loss_fn，也没有断开求导所需的数组关系。其他读取用 `.shape` 或打印已要求完成的参数数组，不再次调用模型、loss或更新。


In [ ]:
# ONE PREPARED CALL — NOT executed in this material-preparation turn.

loss_before_update = train_step(
    model,
    X,
    Y,
    loss_and_grad_fn,
    optimizer,
)

print("loss BEFORE this update:", loss_before_update.item())
print("optimizer step AFTER this update:", optimizer.state["step"].item())
print("parameter leaf shape:", model.head["bias"].shape)
print("parameters AFTER this update:", model.head["bias"])

# Hand-derived expectations: returned loss=ln(3), updated step=1,
# updated parameters approximately [-0.001, -0.001, +0.001] at head.bias.
# This cell has no recorded output; it is not a runtime test or a training run.


## 本节的Shape、工作副本顺序与停止点

| 对象 | 本例shape或结构 |
|---|---|
| X、已shift的Y | `(2,2)`、`(2,2)` |
| raw logits | `(2,2,3)` |
| flat logits / targets | `(4,3)` / `(4,)` |
| token losses / scalar mean loss | `(4,)` / `()` |
| 参数与grads对应叶子 | `(3,)` / `(3,)` |
| Adam m、v叶子 | `(3,)` / `(3,)` |
| step与learning_rate | `()` / `()` |

之后需要使用work副本时：

1. 在仓库根目录 `git pull`，用时间戳新文件名和 `cp -n` 将76母版复制到已有 `notebooks/work/`；只打开新副本。
2. 选择项目Python3.12 / `.venv` kernel，从imports、完整toy类与X/Y开始，再定义loss_fn。
3. 阅读逆向求导与函数变换说明，理解梯度如何按参数树返回；阅读Adam更新接口后，使用本节一整组初始化定义。
4. 阅读显式求值边界和loss前后时点，再定义train_step；末尾只准备一次调用与返回后的日志。

如果以后再次运行调用单元，会继续推进模型参数与优化器历史，不能再套用“初始零参数”的首步手算。要恢复这个首步起点，应重新建立本节同一组model、loss_and_grad_fn、optimizer并初始化；只换model而保留旧求导包装器，会使模型实例不匹配。这里说明使用语义，不安排重复运行或恢复实验。

本节完成大Step17末节的材料准备，不能据此写成学员已经训练、掌握课程或完成运行验证。17.1–17.4的独立材料验收保留，17.5当前是初始材料发布状态。

累计实际修正：16.1=**1/2**、16.2=**1/2**（含已记录的超时后完整回复补齐），16.3至16.5及17.1至17.4各**0/2**，17.5初始生成=**0/2**。每小节最多两次实际修正，跨轮与换对话不重置。

**本轮只编写发布17.5，完成即停在大Step17，不进入大Step18。** 没有执行Notebook、测试、训练、实验、采样或用户work副本，没有读取/下载语料，没有代答练习；不返工已合格材料。后续按请求逐小节至大Step20全部完成后停止。


## 参考与课程衔接

- [17.3：nn.value_and_grad母版](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/5979af5ee2c9508c6840b7c02e917ebf403096e6/notebooks/74_step17_3_nn_value_and_grad.ipynb) 与 [17.4：Adam母版](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/5979af5ee2c9508c6840b7c02e917ebf403096e6/notebooks/75_step17_4_adam_optimizer.ipynb)：本节恢复的参数树、loss、梯度和显式Adam设置。
- [MLX v0.32.2 nn/utils.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/utils.py)：Module-aware求导变换。
- [MLX v0.32.2 optimizers.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/optimizers/optimizers.py) 与 [Module实现](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/layers/base.py)：Adam、优化器update与参数引用替换。
- [MLX v0.32.2 Python transforms](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/transforms.cpp) 与 [核心transforms](https://github.com/ml-explore/mlx/blob/v0.32.2/mlx/transforms.cpp)：eval接收数组树、完成指定结果，以及逆向求导的固定版本依据。
- [MLX v0.32.2 scalar conversion](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/convert.cpp)：item转换与求值语义。
- 官方接口：[nn.value_and_grad](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.nn.value_and_grad.html)、[Optimizer.update](https://ml-explore.github.io/mlx/build/html/python/optimizers/_autosummary/mlx.optimizers.Optimizer.update.html)、[mx.eval](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.eval.html)、[array.item](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.array.item.html)。在线页面可能更新，课程行为以v0.32.2源码为准。
- [Course roadmap](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/course-roadmap.md) 与 [teaching handoff](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/teaching-handoff.md)：当前材料状态、累计计数及大Step17停止边界。
